# Movie Review Sentiment Classifier

This notebook trains a simple model that labels movie reviews as **positive** or **negative**.

**Goals**
1. Load and inspect a small labelled dataset
2. Turn text into numbers with TF-IDF
3. Train and evaluate a Logistic Regression classifier
4. Try the model on new reviews

The dataset is included in the notebook, so no download is needed.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42

## 1. Load the data

A small hand-written sample. Replace it with a real dataset when you have one.

In [ ]:
reviews = [
    "An absolutely wonderful film with brilliant acting",
    "I loved every minute, the story was touching and funny",
    "Beautiful visuals and a heartfelt soundtrack",
    "A gripping thriller that kept me on the edge of my seat",
    "Smart, warm, and surprisingly moving",
    "The best comedy I have seen this year",
    "Great performances and a satisfying ending",
    "Charming characters and a delightful script",
    "Fresh, funny, and full of heart",
    "Engaging from start to finish, highly recommended",
    "Stunning cinematography and a memorable score",
    "A joyful film that left me smiling",
    "Boring plot and wooden acting",
    "I wanted to leave the theatre after the first hour",
    "Predictable, clichéd, and far too long",
    "The dialogue was painful and the pacing was terrible",
    "A complete waste of time and money",
    "Flat characters and a confusing story",
    "The special effects looked cheap and unfinished",
    "I fell asleep twice, it was that dull",
    "Awful script with no emotional impact",
    "Disappointing sequel that adds nothing new",
    "Poorly edited and painfully slow",
    "Would not recommend, the jokes never landed",
]
labels = ["positive"] * 12 + ["negative"] * 12

df = pd.DataFrame({"review": reviews, "sentiment": labels})
print(df.shape)
df.head()

In [ ]:
df["sentiment"].value_counts().plot(kind="bar", title="Class balance", rot=0)
plt.ylabel("Number of reviews")
plt.show()

## 2. Split into training and test sets

The test set is held back so we can measure how well the model generalises.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"], df["sentiment"],
    test_size=0.25,
    stratify=df["sentiment"],
    random_state=RANDOM_STATE,
)
print(f"Training examples: {len(X_train)}")
print(f"Test examples:     {len(X_test)}")

## 3. Build and train the model

A pipeline turns text into TF-IDF features, then fits Logistic Regression.

In [ ]:
model = make_pipeline(
    TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2)),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
model.fit(X_train, y_train)
print("Model trained.")

## 4. Evaluate

We look at accuracy, a per-class report, and a confusion matrix.

In [ ]:
y_pred = model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.2f}\n")
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion matrix")
plt.show()

## 5. Try new reviews

Add your own sentences below and run the cell to see the predictions.

In [ ]:
new_reviews = [
    "This film was a pleasure to watch",
    "Terrible acting and a boring story",
    "Not bad, but I expected more",
]

predictions = model.predict(new_reviews)
probabilities = model.predict_proba(new_reviews)

for text, label, probs in zip(new_reviews, predictions, probabilities):
    confidence = probs.max()
    print(f"{label:>8} ({confidence:.0%})  {text}")

## 6. Notes and next steps

- **Small data:** with only 24 reviews, the accuracy score is noisy. Treat it as a demonstration, not a benchmark.
- **Ideas to extend:**
  - Use a real dataset such as the IMDb reviews dataset (store it in `data/` or download it in code)
  - Compare Logistic Regression with Naive Bayes or a linear SVM
  - Use cross-validation instead of a single train/test split
  - Inspect which words the model weighs most heavily